<a href="https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Question and label. Using March 2026 page signals, can I predict which pages will have a CTR below their position tier’s median in April 2026? The label is computed from April data only, and all features come from March, so there is no overlap between the feature window and the target window. June stays sealed as a final test month.

Method. I will use logistic regression as an interpretable first model and random forest as a nonlinear comparison, both validated with grouped validation (whole clients held out). I will then run permutation importance on the better model to see which signals matter on unseen clients.

Why this fits my lane. My lane is about which signals are associated with CTR and engagement. Logistic regression gives readable coefficients, the random forest tests whether interactions between signals (for example, position and page age) add anything, and permutation importance turns the result into a ranked signal report. I am not using gradient boosting because the added complexity is unlikely to earn its keep on a single-month feature set, and a model that is hard to explain does not serve the lane.

Comparison. The week 4 baseline (rank by March CTR gap) is scored on the same April label, the same held-out clients, and the same metric. I will count the model as better only if it wins by a margin I set in advance, and I will report honestly if it does not.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: time-aware and grouped by client.

Time-aware. Features come from March 2026 and the label comes from April 2026, so the feature window ends before the target window starts and nothing from the future leaks into the inputs. June 2026 stays sealed as a final test month that I do not touch here.

Grouped by client. I hold out whole clients, using a 70/30 split by client_hash_id. Pages from the same client share traits (site structure, audience, tracking setup), so a random page-level split would let the model recognize a client it already saw and make the test too easy. With client holdout, the test measures how well the model does on clients it has never seen.

Same split for the baseline. The week 4 rule and the model are scored on exactly the same held-out pages, with the same label and the same metric, so the comparison is fair.

Limits. Only pages that appear in both months with at least 200 impressions are kept, so the slice is smaller than week 4’s and favors stable, visible pages. One client may dominate the test set, so I report how many clients and pages each side contains.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MIN_IMPRESSIONS = 200

def month_page(m):
    return con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= {MIN_IMPRESSIONS}
    """).df()

def add_tier_ctr(df):
    df["ctr"] = df["clicks"] / df["impressions"]
    df["tier"] = pd.cut(df["avg_position"], [0, 3, 10, 20, np.inf],
                        labels=["pos 1-3", "pos 4-10", "pos 11-20", "pos 21+"])
    df["tier_median_ctr"] = df.groupby("tier", observed=True)["ctr"].transform("median")
    return df

mar = add_tier_ctr(month_page("2026-03"))
apr = add_tier_ctr(month_page("2026-04"))

keys = ["client_hash_id", "content_hash_id"]
frame = mar.merge(apr, on=keys, suffixes=("_mar", "_apr"))

# Label (April only): CTR below the median of its April position tier
frame["label"] = (frame["ctr_apr"] < frame["tier_median_ctr_apr"]).astype(int)

# Week 4 baseline score (March only): how far CTR sat below its March tier median
frame["baseline_score"] = frame["tier_median_ctr_mar"] - frame["ctr_mar"]

# Grouped split: whole clients held out
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(frame, frame["label"], groups=frame["client_hash_id"]))
train, test = frame.iloc[train_idx].copy(), frame.iloc[test_idx].copy()

print("Pages in both months:", len(frame))
print("Train: pages", len(train), "| clients", train["client_hash_id"].nunique(), "| label rate", round(train["label"].mean(), 3))
print("Test:  pages", len(test),  "| clients", test["client_hash_id"].nunique(),  "| label rate", round(test["label"].mean(), 3))
print("Client overlap:", set(train["client_hash_id"]) & set(test["client_hash_id"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in both months: 72383
Train: pages 49949 | clients 26 | label rate 0.362
Test:  pages 22434 | clients 12 | label rate 0.471
Client overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

All three methods are scored on the same held-out clients, the same April label, and the same metrics. About 47% of test pages carry the label, so that is the precision a random list would reach. The week 4 baseline, which only uses March CTR gap, is already strong (AUC 0.753), because pages with low CTR tend to stay low. Logistic regression improves AUC by 0.032 but does worse than the baseline at the top of the list (precision@100 of 0.68 versus 0.77), so I do not count it as a clear win. The random forest is the only method that clearly beats the baseline, improving AUC by 0.059 and reaching a precision@100 of 0.98. This suggests nonlinear interactions between position, impressions, and CTR add information that a straight rule and a linear model miss. I treat this as one split and one seed, so I will check how stable it is before claiming it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def make_X(d):
    return pd.DataFrame({
        "log_impressions": np.log1p(d["impressions_mar"]),
        "avg_position": d["avg_position_mar"],
        "ctr_mar": d["ctr_mar"],
        "tier_median_ctr": d["tier_median_ctr_mar"],
    })

X_train, X_test = make_X(train), make_X(test)
y_train, y_test = train["label"], test["label"]

def precision_at_k(y, score, k=100):
    top = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[top].mean())

results = []

# Week 4 baseline: rank by March CTR gap (no training)
b = test["baseline_score"].values
results.append(("Week 4 baseline (March CTR gap)", roc_auc_score(y_test, b),
                average_precision_score(y_test, b), precision_at_k(y_test, b)))

# Logistic regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(X_train, y_train)
p = lr.predict_proba(X_test)[:, 1]
results.append(("Logistic regression", roc_auc_score(y_test, p),
                average_precision_score(y_test, p), precision_at_k(y_test, p)))

# Random forest
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
rf.fit(X_train, y_train)
p = rf.predict_proba(X_test)[:, 1]
results.append(("Random forest", roc_auc_score(y_test, p),
                average_precision_score(y_test, p), precision_at_k(y_test, p)))

table = pd.DataFrame(results, columns=["method", "ROC AUC", "avg precision", "precision@100"]).round(3)
print("Test label rate (the bar for precision):", round(y_test.mean(), 3))
table


Test label rate (the bar for precision): 0.471


,method,ROC AUC,avg precision,precision@100
0,Week 4 baseline (March CTR gap),0.753,0.692,0.77
1,Logistic regression,0.785,0.703,0.68
2,Random forest,0.812,0.771,0.98


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What the model leans on. Permutation importance on held-out clients shows that March CTR matters by far the most: shuffling it drops AUC by 0.238. Average position comes next (0.081), then log impressions (0.025) and the tier median CTR (0.013). So the random forest mainly relies on a page’s own March CTR, which is the same idea the week 4 baseline uses. Its extra gain over the baseline comes from combining CTR with position and volume, not from a new kind of signal.

Where it is wrong. On 22,434 test pages the model is right about 73% of the time (16,434 correct). Errors split into 3,244 false positives and 2,756 false negatives, so it slightly over-flags pages as low-CTR. Error rates are highest for pages at positions 11-20 (39%) and lowest for positions 21+ (11%). They fall as volume rises: about 29% to 32% for low and mid-low volume pages versus 19% for high-volume pages, which fits the idea that CTR is noisier when impressions are few. Pages that moved to a different position tier between March and April are missed more often than pages that stayed (33% versus 25%), which makes sense because their April comparison group differs from the one March describes.

A limit on this result. One client accounts for 17,092 of the 22,434 test pages (about 76%), so these error rates mostly describe that client. The held-out split is grouped by client, but the test set is not balanced across clients, so I would not claim the model works equally well on every client.

What I would not claim. These results come from one training month, one label month, and one split. They show which signals help predict April CTR gaps, not why CTR is low, and not that changing any signal would improve CTR.

In [3]:
from sklearn.inspection import permutation_importance

# 1) What does the model lean on? Permutation importance on held-out clients
pi = permutation_importance(rf, X_test, y_test, scoring="roc_auc",
                            n_repeats=10, random_state=0, n_jobs=-1)
imp = pd.DataFrame({"feature": X_test.columns,
                    "auc_drop": pi.importances_mean.round(4),
                    "std": pi.importances_std.round(4)}).sort_values("auc_drop", ascending=False)
print(imp)

# 2) Where is it wrong? Use a 0.5 probability cutoff on the test set
err = test.copy()
err["p"] = rf.predict_proba(X_test)[:, 1]
err["pred"] = (err["p"] >= 0.5).astype(int)
err["wrong"] = (err["pred"] != err["label"]).astype(int)
err["error_type"] = np.where(err["wrong"] == 0, "correct",
                     np.where(err["pred"] == 1, "false positive", "false negative"))
print(err["error_type"].value_counts())

# Error rate by position tier, volume bucket, and client
err["volume"] = pd.qcut(err["impressions_mar"], 4, labels=["low", "mid-low", "mid-high", "high"])
print(err.groupby("tier_mar", observed=True)["wrong"].agg(n="size", error_rate="mean").round(3))
print(err.groupby("volume", observed=True)["wrong"].agg(n="size", error_rate="mean").round(3))
print(err.groupby("client_hash_id")["wrong"].agg(n="size", error_rate="mean").sort_values("n", ascending=False).head(5).round(3))

# Do pages that moved across tiers between March and April get missed more?
err["tier_changed"] = (err["tier_mar"] != err["tier_apr"]).astype(int)
print(err.groupby("tier_changed")["wrong"].agg(n="size", error_rate="mean").round(3))

           feature  auc_drop     std
2          ctr_mar    0.2377  0.0026
1     avg_position    0.0810  0.0011
0  log_impressions    0.0245  0.0008
3  tier_median_ctr    0.0128  0.0008
error_type
correct           16434
false positive     3244
false negative     2756
Name: count, dtype: int64
               n  error_rate
tier_mar                    
pos 1-3     3403       0.257
pos 4-10   13438       0.272
pos 11-20   3079       0.391
pos 21+     2514       0.107
             n  error_rate
volume                    
low       5615       0.292
mid-low   5609       0.315
mid-high  5603       0.269
high      5607       0.193
                             n  error_rate
client_hash_id                            
client_73cda7b4e4f265ea  17092       0.274
client_e5c2aa26a8598242   2518       0.203
client_3f0ce4d44fe94f3d   1729       0.288
client_3197e6291363b4db    388       0.245
client_65de48885f4ef01b    388       0.286
                  n  error_rate
tier_changed                   
0    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.